# 종합 실습_모델개발 및 최적화 : 은행 고객 이탈 예측 (누수 점검 반영본)
- 목표 : Validation / Test 평균 AUC 최대화
- 최종 모델 : 아래 11단계에서 **독립 CV 점수로 자동 선택** (XGBoost / CatBoost / Soft Voting 앙상블 중)

**누수 방지 원칙**
1. 기준값이 필요한 전처리(결측 중앙값 대체)는 `Pipeline` 안에 넣어, CV의 각 fold train 부분에서만 fit
2. 변수선택·모델비교·튜닝 단계에서는 **CV와 Valid만 출력**, Test는 마지막 12단계에서 **1회만** 평가
3. 선택 편향 완화 : 단계별로 **서로 다른 CV 분할** 사용  
   - 변수선택/모델후보 비교 : `random_state=0` / 튜닝 : `random_state=42` / 최종 모델 선택 : `random_state=2026` (앞 단계에서 한 번도 쓰지 않은 분할)

In [1]:
!pip install xgboost catboost optuna   # Colab 등 미설치 환경에서 실행
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, json
from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
SEED = 42

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 13.8 MB/s eta 0:00:00


## 1. 데이터 불러오기 (필수 - 제시 코드 그대로)

In [2]:
df = pd.read_csv("/content/bank_churn_train.csv", encoding="cp949")

## 2. Data Partition 6:2:2 (필수 - 제시 코드 그대로)

In [3]:
# 1단계: train 60% / (valid + test) 40%
df_train, temp = train_test_split(df, test_size=0.4, random_state=42)

# 2단계: temp를 valid 50% / test 50% → 전체 기준 각 20%
df_valid, df_test = train_test_split(temp, test_size=0.5, random_state=42)

In [4]:
# X, Y 분리 (Target 'Exited'는 변환하지 않음)
X_train, Y_train = df_train.drop('Exited', axis=1), df_train['Exited']
X_valid, Y_valid = df_valid.drop('Exited', axis=1), df_valid['Exited']
X_test,  Y_test  = df_test.drop('Exited', axis=1),  df_test['Exited']
print(X_train.shape, X_valid.shape, X_test.shape)

(1260, 13) (420, 13) (420, 13)


## 3. 데이터 점검 (train 기준)
- `CustomerId` 전부 고유 → 1행 = 고객 1명 (시간/그룹 분할 불필요), `Surname` 식별자, `baseDate` 단일값(상수)
- 결측 : `Geography`(약 2%), `EstimatedSalary`(약 4%) → 5% 미만이므로 단순 대체

In [5]:
print(X_train.nunique()); print('\n결측:\n', X_train.isna().sum()[lambda s: s>0])
print('\n이탈률(train):', Y_train.mean().round(4))

CustomerId            1260
Surname                863
CreditScore            374
Geography                3
Gender                   2
Age                     59
Balance                782
NumOfProducts            4
HasCrCard                2
IsActiveMember           2
EstimatedSalary       1213
baseDate                 1
accountOpeningDate    1076
dtype: int64

결측:
 Geography          25
EstimatedSalary    47
dtype: int64

이탈률(train): 0.2175


## 4. Feature Engineering (행 단위 변환 → 분할 후 각 데이터에 개별 적용해도 누수 없음)
1. 파생변수 : `TenureYears`, `BalanceZero`, `BalSalRatio`, `ProdPerTenure`, `AgeActive`, `Age_Prod` (후보, 7단계에서 선택)
2. 식별자·상수·원본 날짜 제거 : `CustomerId`, `Surname`, `baseDate`, `accountOpeningDate`
3. 범주형 결측 : `Geography` 결측 → 'Unknown' 범주 (행 단위, 기준값 불필요)
4. One-Hot Encoding : `Geography`, `Gender` (valid/test는 train 컬럼 구조로 reindex)

※ 수치형 결측(`EstimatedSalary`, 이에 의존하는 `BalSalRatio`)은 **여기서 채우지 않고** 5단계 Pipeline 안에서 train 중앙값으로 대체

In [6]:
def make_features(X):
    X = X.copy()
    base = pd.to_datetime(X['baseDate']); opened = pd.to_datetime(X['accountOpeningDate'])
    X['TenureYears']   = (base - opened).dt.days / 365.25              # 가입 후 경과 연수
    X['BalanceZero']   = (X['Balance'] == 0).astype(int)               # 잔고 0 여부
    X['BalSalRatio']   = X['Balance'] / X['EstimatedSalary']           # 잔고 / 추정연봉 (Salary 결측 시 NaN 유지)
    X['ProdPerTenure'] = X['NumOfProducts'] / (X['TenureYears'] + 1)   # 연당 상품 수
    X['AgeActive']     = X['Age'] * X['IsActiveMember']                # 나이 × 활동여부
    X['Age_Prod']      = X['Age'] * X['NumOfProducts']                 # 나이 × 상품 수
    X = X.drop(columns=['CustomerId', 'Surname', 'baseDate', 'accountOpeningDate'])
    X['Geography'] = X['Geography'].fillna('Unknown')
    return pd.get_dummies(X, columns=['Geography', 'Gender'], dtype=int)

Xtr = make_features(X_train)
Xva = make_features(X_valid).reindex(columns=Xtr.columns, fill_value=0)
Xte = make_features(X_test).reindex(columns=Xtr.columns, fill_value=0)
print(Xtr.shape, '\n', Xtr.isna().sum()[lambda s: s>0])

(1260, 19) 
 EstimatedSalary    47
BalSalRatio        47
dtype: int64


## 5. 누수 방지 Pipeline + 평가 함수
- `SimpleImputer(median)` 을 모델과 한 Pipeline으로 묶음 → `cross_val_score` 시 fold마다 train 부분 중앙값만 사용
- `score()` 는 **CV·Valid만 반환/출력** (Test 미사용)

In [7]:
def make_pipe(clf, scale=False):
    steps = [('imp', SimpleImputer(strategy='median'))]
    if scale: steps.append(('sc', StandardScaler()))
    return Pipeline(steps + [('clf', clf)])

cv_sel   = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=0)     # 변수선택·후보비교
cv_tune  = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)  # XGB 튜닝
cv_tune1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)         # CatBoost 튜닝
cv_final = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=2026)  # 최종 선택 (미사용 분할)

def score(model, cols, name, cv):
    c = cross_val_score(model, Xtr[cols], Y_train, cv=cv, scoring='roc_auc').mean()
    model.fit(Xtr[cols], Y_train)
    v = roc_auc_score(Y_valid, model.predict_proba(Xva[cols])[:, 1])
    print(f"{name:34s} CV={c:.4f}  Valid={v:.4f}")
    return {'CV': c, 'Valid': v}

## 6. Baseline (가이드 코드 : 전체 get_dummies + DecisionTree max_depth=3) — Valid만 확인

In [8]:
Xb_tr = pd.get_dummies(X_train)
Xb_va = pd.get_dummies(X_valid).reindex(columns=Xb_tr.columns, fill_value=0)
Xb_te = pd.get_dummies(X_test).reindex(columns=Xb_tr.columns, fill_value=0)
base_dt = DecisionTreeClassifier(random_state=42, max_depth=3).fit(Xb_tr, Y_train)
print('Baseline 컬럼 수:', Xb_tr.shape[1], '(Surname·날짜 문자열이 원-핫되어 컬럼 폭증)')
print('Baseline Valid AUC:', round(roc_auc_score(Y_valid, base_dt.predict_proba(Xb_va)[:, 1]), 4))

Baseline 컬럼 수: 1953 (Surname·날짜 문자열이 원-핫되어 컬럼 폭증)
Baseline Valid AUC: 0.782


## 7. 모델 후보 비교 (원본 변수 + TenureYears, cv_sel)

In [9]:
DERIVED = ['BalanceZero', 'BalSalRatio', 'ProdPerTenure', 'AgeActive', 'Age_Prod']
BASE = [c for c in Xtr.columns if c not in DERIVED]      # 원본 변수 + TenureYears
xgb0 = XGBClassifier(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=0, n_jobs=4)
cat0 = CatBoostClassifier(iterations=500, learning_rate=0.03, depth=4, random_seed=SEED, verbose=0, thread_count=4)
cands = {
 'LogisticRegression': make_pipe(LogisticRegression(max_iter=2000), scale=True),
 'RandomForest': make_pipe(RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=SEED, n_jobs=-1)),
 'XGBoost': make_pipe(xgb0),
 'CatBoost': make_pipe(cat0),
}
cand_res = {n: score(m, BASE, n, cv_sel) for n, m in cands.items()}
top2 = sorted(cand_res, key=lambda n: cand_res[n]['CV'], reverse=True)[:2]
print('\nCV 상위 2개 모델:', top2)

LogisticRegression                 CV=0.7397  Valid=0.7700
RandomForest                       CV=0.8319  Valid=0.8502
XGBoost                            CV=0.8408  Valid=0.8723
CatBoost                           CV=0.8416  Valid=0.8659

CV 상위 2개 모델: ['CatBoost', 'XGBoost']


## 8. Feature Selection (Ablation, cv_sel = 5-fold×3회 반복 CV, 판단은 CV만 사용)

In [10]:
NO_T = [c for c in BASE if c != 'TenureYears']
trials = {'A. 원본 + TenureYears': BASE,
          'B. A + 파생변수 5종 전체': BASE + DERIVED,
          'C. A - TenureYears': NO_T,
          'D. C + BalanceZero': NO_T + ['BalanceZero']}
abl = {}
for k, cols in trials.items():
    abl[k] = [cross_val_score(make_pipe(m), Xtr[cols], Y_train, cv=cv_sel, scoring='roc_auc').mean() for m in (xgb0, cat0)]
    print(f"{k:26s} XGB={abl[k][0]:.4f}  CAT={abl[k][1]:.4f}  평균={np.mean(abl[k]):.4f}")
best_set = max(abl, key=lambda k: np.mean(abl[k]))
FEATS = trials[best_set]
print(f'\n선택: {best_set} ({len(FEATS)}개 변수)\n', FEATS)

A. 원본 + TenureYears        XGB=0.8408  CAT=0.8416  평균=0.8412
B. A + 파생변수 5종 전체          XGB=0.8412  CAT=0.8388  평균=0.8400
C. A - TenureYears         XGB=0.8469  CAT=0.8504  평균=0.8486
D. C + BalanceZero         XGB=0.8481  CAT=0.8493  평균=0.8487

선택: D. C + BalanceZero (14개 변수)
 ['CreditScore', 'Age', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Geography_Unknown', 'Gender_Female', 'Gender_Male', 'BalanceZero']


## 9. Hyperparameter 최적화 (Optuna TPE = Bayesian Search)
- 목적함수 : 누수 방지 Pipeline의 train CV AUC (cv_tune: 변수선택과 다른 분할)
- XGBoost 50 trials / CatBoost 25 trials, sampler seed=42
- `RUN_TUNING=False` 로 두면 저장된 최적값(BEST_SAVED)으로 바로 재현

In [11]:
# 탐색 결과 저장값 (RUN_TUNING=False 시 사용)
BEST_SAVED = {
 "XGB": {
  "n_estimators": 150,
  "learning_rate": 0.018053688883303755,
  "max_depth": 4,
  "min_child_weight": 2,
  "subsample": 0.7371647681264241,
  "colsample_bytree": 0.5441691724640847,
  "reg_lambda": 0.5465780041012008,
  "gamma": 2.1382809448163558
 },
 "CAT": {
  "iterations": 600,
  "learning_rate": 0.014865388298897463,
  "depth": 3,
  "l2_leaf_reg": 3.9478512832636325,
  "random_strength": 1.6231956082198273,
  "bagging_temperature": 0.0973084142214798
 }
}

In [12]:
RUN_TUNING = True
optuna.logging.set_verbosity(optuna.logging.WARNING)
Xf = Xtr[FEATS]

def obj_xgb(t):
    p = dict(n_estimators=t.suggest_int('n_estimators', 100, 800, step=50),
             learning_rate=t.suggest_float('learning_rate', 0.01, 0.1, log=True),
             max_depth=t.suggest_int('max_depth', 2, 6), min_child_weight=t.suggest_int('min_child_weight', 1, 10),
             subsample=t.suggest_float('subsample', 0.6, 1.0), colsample_bytree=t.suggest_float('colsample_bytree', 0.5, 1.0),
             reg_lambda=t.suggest_float('reg_lambda', 0.1, 10, log=True), gamma=t.suggest_float('gamma', 0, 5))
    m = make_pipe(XGBClassifier(**p, random_state=SEED, verbosity=0, n_jobs=4))
    return cross_val_score(m, Xf, Y_train, cv=cv_tune, scoring='roc_auc').mean()

def obj_cat(t):
    p = dict(iterations=t.suggest_int('iterations', 200, 1000, step=100),
             learning_rate=t.suggest_float('learning_rate', 0.01, 0.1, log=True), depth=t.suggest_int('depth', 3, 7),
             l2_leaf_reg=t.suggest_float('l2_leaf_reg', 1, 10, log=True),
             random_strength=t.suggest_float('random_strength', 0.1, 5, log=True),
             bagging_temperature=t.suggest_float('bagging_temperature', 0, 1))
    m = make_pipe(CatBoostClassifier(**p, random_seed=SEED, verbose=0, thread_count=4))
    return cross_val_score(m, Xf, Y_train, cv=cv_tune1, scoring='roc_auc').mean()

if RUN_TUNING:
    best = {}
    for name, obj, n in (('XGB', obj_xgb, 50), ('CAT', obj_cat, 25)):
        st = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
        st.optimize(obj, n_trials=n)
        best[name] = st.best_params; print(name, '튜닝 CV AUC =', round(st.best_value, 4))
else:
    best = BEST_SAVED
print(json.dumps(best, indent=1))

XGB 튜닝 CV AUC = 0.8553
CAT 튜닝 CV AUC = 0.8577
{
 "XGB": {
  "n_estimators": 150,
  "learning_rate": 0.018053688883303755,
  "max_depth": 4,
  "min_child_weight": 2,
  "subsample": 0.7371647681264241,
  "colsample_bytree": 0.5441691724640847,
  "reg_lambda": 0.546578004101201,
  "gamma": 2.1382809448163558
 },
 "CAT": {
  "iterations": 600,
  "learning_rate": 0.014865388298897463,
  "depth": 3,
  "l2_leaf_reg": 3.9478512832636325,
  "random_strength": 1.6231956082198276,
  "bagging_temperature": 0.0973084142214798
 }
}


## 10. 최종 후보 구성
Soft Voting 앙상블의 각 구성원도 Pipeline이므로 결측 대체가 각자 train 기준으로 수행됨

In [13]:
xgb_best = make_pipe(XGBClassifier(**best['XGB'], random_state=SEED, verbosity=0, n_jobs=4))
cat_best = make_pipe(CatBoostClassifier(**best['CAT'], random_seed=SEED, verbose=0, thread_count=4))
ens = VotingClassifier([('xgb', xgb_best), ('cat', cat_best)], voting='soft')
finalists = {'XGBoost(tuned)': xgb_best, 'CatBoost(tuned)': cat_best, 'XGB+CAT Soft Voting': ens}

## 11. 최종 모델 선택 (cv_final : 앞 단계에서 쓰지 않은 새 CV 분할 → 선택 편향 완화)
선택 규칙 : **cv_final CV AUC 최고 모델** (Test는 보지 않음)

In [14]:
final_res = {n: score(m, FEATS, n, cv_final) for n, m in finalists.items()}
final_name = max(final_res, key=lambda n: final_res[n]['CV'])
model = finalists[final_name]           # score()에서 train으로 학습 완료
print('\n▶ 최종 선택 모델:', final_name)

XGBoost(tuned)                     CV=0.8549  Valid=0.8774
CatBoost(tuned)                    CV=0.8595  Valid=0.8802
XGB+CAT Soft Voting                CV=0.8586  Valid=0.8808

▶ 최종 선택 모델: CatBoost(tuned)


## 12. 최종 평가 (Test는 이 셀에서 단 1회 사용)

In [15]:
def evaluate(Y_actual, Y_pred, Y_prob, dataset_name):
    r = {'Accuracy': accuracy_score(Y_actual, Y_pred), 'F1': f1_score(Y_actual, Y_pred),
         'Recall': recall_score(Y_actual, Y_pred), 'Precision': precision_score(Y_actual, Y_pred),
         'AUC': roc_auc_score(Y_actual, Y_prob)}
    print(f"\n=== {dataset_name} 평가 ===")
    for k, v in r.items(): print(f"{k:9s}: {v:.4f}")
    return r

Y_valid_pred, Y_valid_prob = model.predict(Xva[FEATS]), model.predict_proba(Xva[FEATS])[:, 1]
Y_test_pred,  Y_test_prob  = model.predict(Xte[FEATS]), model.predict_proba(Xte[FEATS])[:, 1]
valid_scores = evaluate(Y_valid, Y_valid_pred, Y_valid_prob, "Validation")
test_scores  = evaluate(Y_test,  Y_test_pred,  Y_test_prob,  "Test")
print(f"\n평균 AUC = {(valid_scores['AUC'] + test_scores['AUC'])/2:.4f}")
print('참고) Baseline Test AUC =', round(roc_auc_score(Y_test, base_dt.predict_proba(Xb_te)[:, 1]), 4))


=== Validation 평가 ===
Accuracy : 0.8119
F1       : 0.5153
Recall   : 0.3962
Precision: 0.7368
AUC      : 0.8802

=== Test 평가 ===
Accuracy : 0.8690
F1       : 0.4554
Recall   : 0.3382
Precision: 0.6970
AUC      : 0.8652

평균 AUC = 0.8727
참고) Baseline Test AUC = 0.7625


## 13. 참고 : 변수 중요도 (도메인 상식 검증)
※ 12단계 이후의 사후 확인용이며 어떤 선택에도 사용하지 않음

In [16]:
xgb_best.fit(Xtr[FEATS], Y_train); cat_best.fit(Xtr[FEATS], Y_train)
imp = pd.DataFrame({'XGB': xgb_best.named_steps['clf'].feature_importances_,
                    'CAT': cat_best.named_steps['clf'].get_feature_importance() / 100}, index=FEATS)
imp.sort_values('CAT', ascending=False).round(3)

,XGB,CAT
NumOfProducts,0.198,0.337
Age,0.159,0.282
IsActiveMember,0.164,0.160
Balance,0.048,0.066
EstimatedSalary,0.041,0.044
CreditScore,0.039,0.039
Geography_Germany,0.062,0.024
Gender_Male,0.042,0.017
HasCrCard,0.027,0.007
Geography_France,0.059,0.007
